# ChromaDB PDF RAG System

This notebook extracts text from PDFs in `./pdfs`, creates a persistent **ChromaDB** vector database, and allows you to query it via **Ollama** in separate cell blocks.

### Cell 1: Setup & Configuration *(always run first)*

In [2]:
import os
import re
import glob
import json
import requests
import pypdf
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

# --- Config ---
PDF_DIR        = "./pdfs"
CHROMA_DB_DIR  = "./chroma_db"
COLLECTION_NAME = "pdf_rag_docs"
OLLAMA_URL     = "https://aud-thomas-citizen-inspector.trycloudflare.com/api/generate"
MODEL_NAME     = "gemma3:4b"

# Shared ChromaDB client & collection (populated in Cell 2)
chroma_client  = None
collection     = None

print("Config loaded.")

Config loaded.


### Cell 2: Extract PDFs & Build Vector Database *(run once to index)*

Uses **sentence-aware chunking** — splits on natural sentence endings (`. ! ?`) instead of raw character counts. This keeps each chunk semantically complete and improves retrieval accuracy.

If the collection already exists on disk, it simply loads it — no re-indexing.

In [3]:
# ─── Sentence-aware chunking (no NLTK needed) ─────────────────────────────────

def split_into_sentences(text):
    """Split text on sentence boundaries using regex."""
    # Split after . ! ? that are followed by whitespace or end-of-string
    pattern = re.compile(r'(?<=[.!?])\s+')
    return [s.strip() for s in pattern.split(text) if s.strip()]

def sentence_aware_chunks(text, max_chars=600, overlap_sentences=2):
    """
    Groups sentences into chunks that stay under max_chars.
    Overlaps by reusing the last `overlap_sentences` of the previous chunk
    so context is not lost at boundaries.
    """
    sentences = split_into_sentences(text)
    chunks = []
    current = []
    current_len = 0

    for sent in sentences:
        if current_len + len(sent) > max_chars and current:
            chunks.append(' '.join(current))
            # Carry over last N sentences as overlap for continuity
            current = current[-overlap_sentences:]
            current_len = sum(len(s) for s in current)
        current.append(sent)
        current_len += len(sent)

    if current:
        chunks.append(' '.join(current))

    return chunks

def extract_chunks_from_pdfs(pdf_folder, max_chars=600, overlap_sentences=2):
    pdf_files = glob.glob(os.path.join(pdf_folder, '*.pdf'))
    if not pdf_files:
        print(f'Warning: No PDF files found in {pdf_folder}')
        return []

    all_chunks, chunk_id = [], 0
    for pdf_path in pdf_files:
        file_name = os.path.basename(pdf_path)
        print(f'  Processing: {file_name}')
        try:
            reader = pypdf.PdfReader(pdf_path)
            for page_num, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or '').strip()
                if not text:
                    continue
                page_chunks = sentence_aware_chunks(text, max_chars, overlap_sentences)
                for chunk_idx, chunk_text in enumerate(page_chunks):
                    if chunk_text:
                        all_chunks.append({
                            'id': f'chunk_{chunk_id}',
                            'text': chunk_text,
                            'metadata': {
                                'source': file_name,
                                'page': page_num,
                                'chunk_index': chunk_idx
                            }
                        })
                        chunk_id += 1
        except Exception as e:
            print(f'  Error reading {file_name}: {e}')

    print(f'Extracted {len(all_chunks)} sentence-aware chunks from {len(pdf_files)} PDF(s).')
    return all_chunks

# ─── Build / Load ChromaDB ────────────────────────────────────────────────────
print('Setting up ChromaDB...')
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
ef = DefaultEmbeddingFunction()  # all-MiniLM-L6-v2 via ONNX

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=ef
)

if collection.count() == 0:
    chunks = extract_chunks_from_pdfs(PDF_DIR)
    print(f'Indexing {len(chunks)} chunks into ChromaDB...')
    batch_size = 50
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        collection.add(
            ids=[c['id'] for c in batch],
            documents=[c['text'] for c in batch],
            metadatas=[c['metadata'] for c in batch]
        )
        print(f"  Indexed batch {i // batch_size + 1}/{(len(chunks) + batch_size - 1) // batch_size}")
    print('Indexing complete!')
else:
    print(f"Collection '{COLLECTION_NAME}' already exists with {collection.count()} chunks — skipping re-index.")

Setting up ChromaDB...
  Processing: 02_Chapter_01_Arjuna's_Distress.pdf
  Processing: 03_Chapter_02_The_Path_of_Doctrines.pdf
Extracted 482 sentence-aware chunks from 2 PDF(s).
Indexing 482 chunks into ChromaDB...
  Indexed batch 1/10
  Indexed batch 2/10
  Indexed batch 3/10
  Indexed batch 4/10
  Indexed batch 5/10
  Indexed batch 6/10
  Indexed batch 7/10
  Indexed batch 8/10
  Indexed batch 9/10
  Indexed batch 10/10
Indexing complete!


### Cell 3: Query the RAG System *(run independently, anytime)*
Change `user_query` below and run this cell to ask any question about your PDFs.

In [8]:
def _load_collection_if_needed():
    """Auto-connects to the persisted ChromaDB if collection is not already in memory."""
    global collection, chroma_client
    if collection is not None:
        return
    print("Re-connecting to ChromaDB from disk...")
    chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
    ef = DefaultEmbeddingFunction()
    collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME, embedding_function=ef)
    print(f"Loaded collection with {collection.count()} chunks.")

def query_ollama(prompt, system_context="", model=MODEL_NAME, timeout=120):
    full_prompt = f"""You are an intelligent assistant. Answer the user's question accurately using ONLY the provided context below.
If the context does not contain enough information to answer, clearly state that.

--- RETRIEVED PDF CONTEXT ---
{system_context}
-----------------------------

Question: {prompt}

Answer:"""
    try:
        response = requests.post(OLLAMA_URL, json={"model": model, "prompt": full_prompt, "stream": False}, timeout=timeout)
        response.raise_for_status()
        try:
            data = response.json()
        except json.JSONDecodeError:
            lines = [l.strip() for l in response.text.strip().splitlines() if l.strip()]
            data = json.loads(lines[-1])
        return data.get("response", str(data)) if isinstance(data, dict) else str(data)
    except Exception as e:
        return f"Error connecting to Ollama: {e}"

def ask_question(query, n_results=4):
    _load_collection_if_needed()

    print(f"\nSearching ChromaDB for: '{query}'...")
    results = collection.query(query_texts=[query], n_results=n_results)

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    if not documents:
        print("No relevant context found in PDFs.")
        return

    print("\n--- RETRIEVED SOURCES ---")
    context_blocks = []
    for idx, (doc, meta) in enumerate(zip(documents, metadatas), start=1):
        source_info = f"Source: {meta.get('source', 'Unknown')} (Page {meta.get('page', '?')}, Chunk {meta.get('chunk_index', '?')})"
        print(f"[{idx}] {source_info}")
        print(f"    Excerpt: {doc[:140]}...\n")
        context_blocks.append(f"[{source_info}]\n{doc}")

    print("Querying Ollama LLM...")
    answer = query_ollama(query, system_context="\n\n".join(context_blocks))
    print("\n--- RAG ANSWER ---")
    print(answer)
    print("=" * 60)

# Change your question here and re-run this cell:
user_query = " rjuna now summarizes his arguments for staying away from the war."
ask_question(user_query)


Searching ChromaDB for: ' rjuna now summarizes his arguments for staying away from the war.'...

--- RETRIEVED SOURCES ---
[1] Source: 02_Chapter_01_Arjuna's_Distress.pdf (Page 22, Chunk 5)
    Excerpt: Arju na was not afraid of fighting as he had fought 
many wars before and  had won them. But this time it was different. His 
enemies were p...

[2] Source: 03_Chapter_02_The_Path_of_Doctrines.pdf (Page 19, Chunk 0)
    Excerpt: Chapter 2                                                                                                                                   ...

[3] Source: 02_Chapter_01_Arjuna's_Distress.pdf (Page 18, Chunk 4)
    Excerpt: (1.45)  
 
It would be better for me if I relinquish ed my weapons and offer ed 
no resistance while Dhrutarashtra’s sons kill me in this wa...

[4] Source: 02_Chapter_01_Arjuna's_Distress.pdf (Page 16, Chunk 3)
    Excerpt: It is his duty  to preserve 
and protect the innocent in society. While A rjuna is aware that the 
Kauravas deserve t